# Blog-to-Social Content Converter

## Project Overview

This notebook converts **one long-form blog article** into **five distinct social content formats**, each tailored to the norms, audience, and constraints of its platform:

| Output Format | Constraints | Tone Default |
|---------------|-------------|--------------|
| **Short summary** | 2-3 sentences, platform-agnostic | Neutral / informative |
| **LinkedIn post** | 1 300 chars, professional paragraphs | Authoritative, insight-driven |
| **Twitter / X thread** | 4-6 tweets × 280 chars each | Punchy, curiosity hooks |
| **Email snippet** | Subject ≤ 60 chars + 2-sentence teaser | Urgency + intrigue |
| **Instagram caption** | ≤ 2 200 chars, emoji-rich, hashtags | Casual, visual-first |

**Key engineering pattern — prompt adaptation:** the *same* source content is reshaped by changing a single `tone` / `style` control variable, demonstrating how prompt design drives output quality more than model choice.

```
Blog article (500-2 000 words)
  ├─ extract_thesis()          → core argument
  ├─ extract_key_points()      → supporting evidence
  └─ adapt(platform, tone)     → 5 platform-specific outputs
       ├─ Short summary
       ├─ LinkedIn post
       ├─ Twitter/X thread
       ├─ Email snippet
       └─ Instagram caption
```

**Stack:** Python standard library — no ML frameworks required. Optional Ollama for LLM-enhanced generation.

## Learning Goals

By the end of this notebook you will be able to:

1. **Design platform-aware prompts** — understand why LinkedIn ≠ Twitter ≠ email in tone, length, and hook style
2. **Implement tone/style controls** — switch between *professional*, *casual*, *witty*, and *urgent* with one parameter
3. **Build a content repurposing pipeline** — convert one article into five deliverables automatically
4. **Explain prompt adaptation** — articulate *why* a LinkedIn prompt differs from a Twitter prompt, beyond just length
5. **Evaluate output quality** — check constraint satisfaction (char limits, formatting) and tone consistency
6. **Compare rule-based vs. LLM generation** — understand deterministic extraction versus creative rewriting

## Problem Statement

**The content repurposing bottleneck:** A content team publishes a 1 500-word blog post. They now need:
- A LinkedIn summary for the company page
- A Twitter thread for the founder's account
- An email teaser for the weekly newsletter
- A short summary for internal Slack
- An Instagram caption for the brand account

Doing this manually takes 45-90 minutes *per article*. Multiply by 3 articles/week and you have a full-time content role just for repurposing.

**Why this is hard (not just "make it shorter"):**

| Platform | Audience expects | Hook style | Length |
|----------|-----------------|------------|--------|
| LinkedIn | Professional credibility | "Here's what I learned…" | 200-1 300 chars |
| Twitter/X | Quick insight or hot take | "Thread 🧵" / contrarian opener | 4-6 × 280 chars |
| Email | Curiosity gap | "You won't believe…" / "This changes…" | Subject + 2 sentences |
| Summary | Neutral accuracy | None — just the facts | 2-3 sentences |
| Instagram | Visual storytelling | Emoji + question | ≤ 2 200 chars |

**Our approach:** Rule-based extraction baseline + optional LLM enhancement, with a `tone` parameter that reshapes every output.

## Why This Project Matters

Content repurposing is standard practice at every scale:

- **Solo creators:** maximize reach across platforms without doubling writing time
- **Marketing teams:** maintain brand voice consistency across 5+ channels
- **Companies:** 1 blog → 20+ social touchpoints per week, compounding audience growth
- **Agencies:** scale content operations without proportionally scaling headcount

**The business math:**

| Metric | Manual | Automated |
|--------|--------|-----------|
| Time per article | 60 min | 5 min |
| Articles/week | 3 | 3 |
| Weekly time cost | 3 hours | 15 min |
| Annual savings | — | ~130 hours |

**The prompt engineering lesson:** This project teaches that *prompt design* is the leverage point. A well-structured prompt with clear constraints outperforms a vague "rewrite this for LinkedIn" instruction by an order of magnitude.

## Dataset Overview

We use **three embedded blog articles** covering different topics, tones, and lengths:

| # | Topic | Domain | Words | Original Tone |
|---|-------|--------|-------|---------------|
| 1 | Prompt Engineering for Developers | Tech / Education | ~500 | Instructional |
| 2 | Work-Life Balance Myths | Wellness / Personal | ~450 | Conversational |
| 3 | AI Safety Paradox | AI Policy / Thought Leadership | ~500 | Analytical |

Using diverse source material lets us test whether the pipeline handles different domains and writing styles consistently.

> **Note:** These are self-contained sample articles embedded in the notebook. No external download is required — the articles are defined directly in code cells below.

## Dataset Source & License

These sample blog articles are **original synthetic content** written specifically for this notebook. They are not sourced from any external dataset or copyrighted publication.

- **Source:** Custom-written for educational purposes
- **License:** Public domain / educational use
- **Format:** Python dictionaries with `title`, `body`, `author`, `date` fields

## Environment Setup

We check for Ollama availability. If unavailable, all generation falls back to deterministic rule-based methods — the notebook runs fully offline.

In [1]:
import urllib.request
import json
import re
import textwrap
from typing import Any, Dict, List, Optional


def is_ollama_available(host: str = "http://localhost:11434") -> bool:
    """Check if Ollama server is running and reachable."""
    try:
        with urllib.request.urlopen(f"{host}/api/tags", timeout=3) as resp:
            return resp.status == 200
    except Exception:
        return False


OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen3:8b"
USE_LLM = is_ollama_available(OLLAMA_HOST)

print(f"Ollama available: {USE_LLM}")
if USE_LLM:
    print(f"  Model: {OLLAMA_MODEL}")
else:
    print("  → All generation uses rule-based extraction (deterministic, offline)")


Ollama available: False
  → All generation uses rule-based extraction (deterministic, offline)


## Imports

Only standard library modules are used. No pip installs required.

In [2]:
# All imports already loaded above (urllib, json, re, textwrap, typing)
# Confirming availability:
print("stdlib modules: urllib, json, re, textwrap, typing — ready")


stdlib modules: urllib, json, re, textwrap, typing — ready


## Configuration & Constants

### Tone / Style Controls

The key design insight: **a single `tone` parameter reshapes every output.**

We define four tones. Each changes vocabulary, sentence structure, and hook style:

| Tone | Vocabulary | Sentence Style | Hook Pattern |
|------|-----------|---------------|-------------|
| `professional` | Formal, jargon-ok | Compound, evidence-backed | "Research shows…" |
| `casual` | Colloquial, contractions | Short, punchy | "So here's the thing…" |
| `witty` | Playful, metaphors | Rhetorical questions, callbacks | "Plot twist:…" |
| `urgent` | Action verbs, imperatives | Direct, imperative | "Stop doing X. Start doing Y." |

In [3]:
# ── Tone Definitions ──
TONES = {
    "professional": {
        "opener_templates": [
            "A key insight from recent work: {point}",
            "Here's what the evidence suggests: {point}",
            "{point} — and the implications are significant.",
        ],
        "cta": "What's your perspective? I'd welcome your thoughts.",
        "emoji_density": 0,       # no emoji
        "hashtag_count": 0,
        "formality": "high",
    },
    "casual": {
        "opener_templates": [
            "So here's the thing — {point}",
            "Real talk: {point}",
            "I keep thinking about this: {point}",
        ],
        "cta": "Thoughts? Drop a comment 👇",
        "emoji_density": 1,       # light emoji
        "hashtag_count": 2,
        "formality": "low",
    },
    "witty": {
        "opener_templates": [
            "Plot twist: {point}",
            "Everyone's talking about X. Nobody's talking about {point}",
            "Hot take: {point} (hear me out)",
        ],
        "cta": "Fight me in the comments 😄",
        "emoji_density": 2,       # moderate emoji
        "hashtag_count": 1,
        "formality": "medium",
    },
    "urgent": {
        "opener_templates": [
            "Stop scrolling. {point}",
            "This changes everything: {point}",
            "If you read one thing today: {point}",
        ],
        "cta": "Don't wait — start today.",
        "emoji_density": 1,
        "hashtag_count": 0,
        "formality": "medium",
    },
}

# Platform character limits
PLATFORM_LIMITS = {
    "summary": 500,
    "linkedin": 1300,
    "twitter_tweet": 280,
    "twitter_thread_tweets": 5,
    "email_subject": 60,
    "email_body": 300,
    "instagram": 2200,
}

DEFAULT_TONE = "professional"

print("Tone definitions loaded:", list(TONES.keys()))
print("Platform limits:", json.dumps(PLATFORM_LIMITS, indent=2))


Tone definitions loaded: ['professional', 'casual', 'witty', 'urgent']
Platform limits: {
  "summary": 500,
  "linkedin": 1300,
  "twitter_tweet": 280,
  "twitter_thread_tweets": 5,
  "email_subject": 60,
  "email_body": 300,
  "instagram": 2200
}


## Data Loading — Sample Blog Articles

Three realistic blog articles covering different domains and writing styles.

In [4]:
ARTICLES = [
    {
        "title": "Why Every Developer Should Learn Prompt Engineering in 2025",
        "body": (
            "Prompt engineering is no longer a niche skill. It is becoming as essential as "
            "git for modern developers. Here is why.\n\n"
            "The AI landscape has shifted dramatically. LLMs are getting cheaper, faster, and "
            "more capable. But raw model power means nothing without the ability to extract "
            "value from it. That is where prompt engineering comes in.\n\n"
            "Think of prompts as the new API. Five years ago, developers wrote REST API calls. "
            "Today, they write prompts to GPT, Claude, and custom models. The difference? "
            "Instead of calling a fixed endpoint, you design instructions that shape model "
            "behavior.\n\n"
            "The game-changing insight: small prompt tweaks create massive output differences. "
            "Changing 'explain' to 'explain like I am five' does not just change vocabulary — "
            "it changes reasoning depth, analogy types, and communication style. This is "
            "prompt sensitivity, and it is your superpower.\n\n"
            "Three practical takeaways:\n"
            "1. Prompting is an experimental discipline. You test hypotheses, not guess. "
            "Add a step-by-step instruction and compare output quality.\n"
            "2. Context is leverage. A 2000-token context with background information will "
            "outperform a 10k-token context with noise. Curate inputs ruthlessly.\n"
            "3. Chains beat single prompts. One mega-prompt is fragile. Chain smaller prompts "
            "and each becomes testable, debuggable, and ownable."
        ),
        "author": "Tech Blogger",
        "date": "2025-04-15",
    },
    {
        "title": "The Burnout Myth: Why Work-Life Balance Is Not About Balance",
        "body": (
            "Everyone talks about work-life balance like it is a math equation: if work is "
            "50 percent and life is 50 percent, you are balanced. Wrong.\n\n"
            "The burnout crisis is not about unequal time allocation. It is about misalignment. "
            "The most burned-out people I know work 40 hours a week. The least burned-out work "
            "70. The difference? Fulfillment.\n\n"
            "Work fulfillment comes from three things: autonomy (you make meaningful decisions), "
            "mastery (you are getting visibly better at something), and purpose (the work "
            "connects to your values).\n\n"
            "Life fulfillment comes from relationships (deep connection, not surface socializing), "
            "growth (learning and identity expansion outside work), and rest (genuine recovery, "
            "not Netflix binges).\n\n"
            "Most work-life balance advice attacks the wrong problem. 'Work less' is useless "
            "when your 40 hours are unfulfilling. 'Exercise more' misses the point when your "
            "relationships feel transactional.\n\n"
            "What actually works:\n"
            "1. Audit fulfillment, not hours. Which 10 hours matter most?\n"
            "2. Stack your values. Find work overlapping your values.\n"
            "3. Protect recovery time like a client meeting. Recovery is maintenance, not laziness.\n\n"
            "Stop optimizing for balance. Optimize for alignment."
        ),
        "author": "Wellness Coach",
        "date": "2025-04-14",
    },
    {
        "title": "The AI Safety Paradox: Why Slowing Down Makes Deployment Faster",
        "body": (
            "The AI safety community and the move-fast crowd are in open conflict. One side "
            "says slow down, we do not understand these systems. The other says regulation "
            "kills innovation, let the market sort it out.\n\n"
            "Both miss the paradox: the fastest path to safe, deployed AI is rigorous safety "
            "work up front.\n\n"
            "Here is the asymmetry: a security flaw at deployment costs 10x more to fix than "
            "one found in design review. A capability misalignment discovered in RLHF costs "
            "100x more after 10 million users. Alignment failure after launch can destroy a "
            "company.\n\n"
            "The market punishes unsafe AI, but too late — after reputational damage, regulatory "
            "backlash, and user harm. The unsafe path was not faster; it just shifted costs "
            "from months 1-6 to months 7-12.\n\n"
            "Three concrete takeaways for teams building LLM products:\n"
            "1. Red team during development, not after launch. Find misalignment when fixing "
            "costs a thousand dollars, not a million.\n"
            "2. Safety metrics are leading indicators. Track hallucination rate, refusal "
            "calibration, and adversarial robustness. These predict harm before it happens.\n"
            "3. Deploy slowly with feedback loops. Risk is managed by measurement, not removed "
            "by caution.\n\n"
            "The paradox resolves when you realize safety work is deployment engineering. "
            "It is not a different timeline; it is the critical path."
        ),
        "author": "AI Researcher",
        "date": "2025-04-13",
    },
]

print(f"Loaded {len(ARTICLES)} articles:\n")
for i, a in enumerate(ARTICLES, 1):
    word_count = len(a["body"].split())
    print(f"  {i}. {a['title']}")
    print(f"     Author: {a['author']}  |  Words: {word_count}  |  Date: {a['date']}")


Loaded 3 articles:

  1. Why Every Developer Should Learn Prompt Engineering in 2025
     Author: Tech Blogger  |  Words: 199  |  Date: 2025-04-15
  2. The Burnout Myth: Why Work-Life Balance Is Not About Balance
     Author: Wellness Coach  |  Words: 178  |  Date: 2025-04-14
  3. The AI Safety Paradox: Why Slowing Down Makes Deployment Faster
     Author: AI Researcher  |  Words: 210  |  Date: 2025-04-13


## Data Validation

Basic sanity checks on our source articles before processing.

In [5]:
print("Validating articles...\n")

for i, a in enumerate(ARTICLES, 1):
    issues = []
    if not a.get("title"):
        issues.append("missing title")
    if not a.get("body"):
        issues.append("missing body")
    word_count = len(a["body"].split())
    if word_count < 50:
        issues.append(f"too short ({word_count} words)")
    if word_count > 5000:
        issues.append(f"very long ({word_count} words)")
    if not a.get("author"):
        issues.append("missing author")

    status = "PASS" if not issues else f"WARN: {', '.join(issues)}"
    print(f"  Article {i}: {status}  ({word_count} words)")

print("\nAll articles validated.")


Validating articles...

  Article 1: PASS  (199 words)
  Article 2: PASS  (178 words)
  Article 3: PASS  (210 words)

All articles validated.


## Exploratory Analysis

Let's understand the structure of our source content to inform how we'll extract and adapt it.

In [6]:
print("=" * 70)
print("CONTENT STRUCTURE ANALYSIS")
print("=" * 70)

for i, a in enumerate(ARTICLES, 1):
    body = a["body"]
    paragraphs = [p.strip() for p in body.split("\n\n") if p.strip()]
    sentences = re.split(r"(?<=[.!?])\s+", body)
    sentences = [s for s in sentences if len(s.strip()) > 5]

    # Detect numbered points
    numbered = [s for s in body.split("\n") if re.match(r"^\d+\.", s.strip())]

    # Detect questions
    questions = [s for s in sentences if s.strip().endswith("?")]

    print(f"\nArticle {i}: {a['title'][:50]}...")
    print(f"  Paragraphs:   {len(paragraphs)}")
    print(f"  Sentences:    {len(sentences)}")
    print(f"  Numbered pts: {len(numbered)}")
    print(f"  Questions:    {len(questions)}")
    print(f"  Avg sent len: {sum(len(s.split()) for s in sentences) // max(len(sentences), 1)} words")
    if numbered:
        print(f"  First point:  {numbered[0][:80]}...")


CONTENT STRUCTURE ANALYSIS

Article 1: Why Every Developer Should Learn Prompt Engineerin...
  Paragraphs:   5
  Sentences:    25
  Numbered pts: 3
  Questions:    1
  Avg sent len: 7 words
  First point:  1. Prompting is an experimental discipline. You test hypotheses, not guess. Add ...

Article 2: The Burnout Myth: Why Work-Life Balance Is Not Abo...
  Paragraphs:   7
  Sentences:    22
  Numbered pts: 3
  Questions:    2
  Avg sent len: 8 words
  First point:  1. Audit fulfillment, not hours. Which 10 hours matter most?...

Article 3: The AI Safety Paradox: Why Slowing Down Makes Depl...
  Paragraphs:   6
  Sentences:    19
  Numbered pts: 3
  Questions:    0
  Avg sent len: 10 words
  First point:  1. Red team during development, not after launch. Find misalignment when fixing ...


## Preprocessing — Extracting Key Elements

Before generating platform-specific content, we extract **three reusable elements** from every article:

1. **Thesis** — the single core argument (1 sentence)
2. **Key points** — supporting evidence / takeaways (3-5 items)
3. **Hook candidates** — attention-grabbing phrases for openers

In [7]:
def extract_thesis(body: str) -> str:
    """Extract the core thesis — the most 'claim-like' sentence."""
    sentences = re.split(r"(?<=[.!?])\s+", body.strip())
    sentences = [s.strip() for s in sentences if len(s.split()) >= 6]

    # Score: prefer sentences with claim markers
    claim_markers = ["is", "means", "shows", "proves", "reveals", "because",
                     "the key", "the real", "the truth", "the paradox"]
    scored = []
    for s in sentences:
        score = sum(1 for m in claim_markers if m in s.lower())
        # Bonus for mid-article sentences (thesis rarely in first line)
        idx = body.find(s)
        if 0.1 < idx / max(len(body), 1) < 0.5:
            score += 1
        scored.append((score, s))

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1] if scored else sentences[0]


def extract_key_points(body: str, max_points: int = 5) -> List[str]:
    """Extract key points: numbered items, strong claims, or evidence sentences."""
    lines = body.split("\n")
    points = []

    # First pass: explicit numbered items
    for line in lines:
        line = line.strip()
        if re.match(r"^\d+\.\s", line) and len(line) > 15:
            # Clean up the point text
            clean = re.sub(r"^\d+\.\s*", "", line).strip()
            points.append(clean)

    # Second pass: strong claim sentences (if we need more)
    if len(points) < max_points:
        sentences = re.split(r"(?<=[.!?])\s+", body)
        for s in sentences:
            s = s.strip()
            if len(s.split()) >= 8 and s not in points:
                if any(w in s.lower() for w in ["must", "should", "key", "important",
                                                  "critical", "essential", "never"]):
                    points.append(s)
            if len(points) >= max_points:
                break

    return points[:max_points]


def extract_hooks(title: str, body: str, max_hooks: int = 3) -> List[str]:
    """Extract attention-grabbing phrases for social media openers."""
    hooks = []

    # Title is always a hook candidate
    hooks.append(title)

    # Sentences with contrast or surprise
    sentences = re.split(r"(?<=[.!?])\s+", body)
    for s in sentences:
        s = s.strip()
        if any(w in s.lower() for w in ["wrong", "myth", "paradox", "surprise",
                                         "actually", "but", "however", "plot twist"]):
            if len(s.split()) <= 20:
                hooks.append(s)
        if len(hooks) >= max_hooks:
            break

    return hooks[:max_hooks]


# Test extraction on all articles
for i, a in enumerate(ARTICLES, 1):
    print(f"\n{'=' * 70}")
    print(f"ARTICLE {i}: {a['title']}")
    print(f"{'=' * 70}")

    thesis = extract_thesis(a["body"])
    print(f"\n  THESIS: {thesis[:120]}...")

    points = extract_key_points(a["body"])
    print(f"\n  KEY POINTS ({len(points)}):")
    for j, p in enumerate(points, 1):
        print(f"    {j}. {p[:100]}...")

    hooks = extract_hooks(a["title"], a["body"])
    print(f"\n  HOOKS ({len(hooks)}):")
    for h in hooks:
        print(f"    → {h[:100]}")



ARTICLE 1: Why Every Developer Should Learn Prompt Engineering in 2025

  THESIS: But raw model power means nothing without the ability to extract value from it....

  KEY POINTS (4):
    1. Prompting is an experimental discipline. You test hypotheses, not guess. Add a step-by-step instruct...
    2. Context is leverage. A 2000-token context with background information will outperform a 10k-token co...
    3. Chains beat single prompts. One mega-prompt is fragile. Chain smaller prompts and each becomes testa...
    4. It is becoming as essential as git for modern developers....

  HOOKS (2):
    → Why Every Developer Should Learn Prompt Engineering in 2025
    → But raw model power means nothing without the ability to extract value from it.

ARTICLE 2: The Burnout Myth: Why Work-Life Balance Is Not About Balance

  THESIS: The burnout crisis is not about unequal time allocation....

  KEY POINTS (3):
    1. Audit fulfillment, not hours. Which 10 hours matter most?...
    2. Stack your

## Understanding Prompt Adaptation

### Why does the same content need different prompts per platform?

Prompt adaptation is **not just shortening text**. It requires changing five dimensions simultaneously:

| Dimension | LinkedIn | Twitter/X | Email | Summary | Instagram |
|-----------|----------|-----------|-------|---------|-----------|
| **Length** | 200-1300 chars | 280 chars/tweet | Subject + 2 lines | 2-3 sentences | ≤ 2200 chars |
| **Hook** | Credibility ("I've seen…") | Curiosity ("Thread 🧵") | Urgency ("Don't miss") | None (factual) | Visual ("✨ POV:…") |
| **Tone** | Authoritative | Punchy, opinionated | Personal, direct | Neutral | Playful, emoji-rich |
| **Structure** | Paragraphs with line breaks | Numbered thread tweets | Subject + preview | Single paragraph | Caption + hashtags |
| **CTA** | "What do you think?" | "Follow for more" | "Read the full article" | None | "Save this 🔖" |

### The tone parameter

Each tone (`professional`, `casual`, `witty`, `urgent`) changes the *same* content across *all* platforms. This is the core prompt engineering insight: **tone is orthogonal to platform**. You can be professional on Twitter or casual on LinkedIn.

The tone modifies:
- **Vocabulary:** "leverage" (professional) vs. "use" (casual) vs. "weaponize" (witty)
- **Sentence structure:** compound (professional) vs. fragments (casual)
- **Opener style:** evidence-based (professional) vs. rhetorical (witty) vs. imperative (urgent)
- **Emoji usage:** none (professional) → light (casual) → moderate (witty)

## Baseline — Rule-Based Generation

These generators use **no LLM**. They extract, compress, and reformat using heuristics. This is our baseline to compare against LLM-enhanced versions.

In [8]:
def generate_summary(title: str, body: str, tone: str = DEFAULT_TONE) -> str:
    """Generate a 2-3 sentence platform-agnostic summary."""
    thesis = extract_thesis(body)
    points = extract_key_points(body, max_points=2)

    tone_cfg = TONES[tone]
    if tone_cfg["formality"] == "high":
        summary = f"{title}. {thesis}"
        if points:
            summary += f" Key insight: {points[0][:80]}."
    elif tone_cfg["formality"] == "low":
        summary = f"{title} — {thesis.lower()}"
        if points:
            summary += f" Basically: {points[0][:60]}."
    else:
        opener = tone_cfg["opener_templates"][0].format(point=thesis[:60])
        summary = f"{opener}. {title}."

    # Enforce limit
    if len(summary) > PLATFORM_LIMITS["summary"]:
        summary = summary[:PLATFORM_LIMITS["summary"] - 3] + "..."
    return summary


def generate_linkedin(title: str, body: str, tone: str = DEFAULT_TONE) -> str:
    """Generate a LinkedIn post with paragraphs and CTA."""
    tone_cfg = TONES[tone]
    thesis = extract_thesis(body)
    points = extract_key_points(body, max_points=3)
    hooks = extract_hooks(title, body)

    # Build opener
    opener_tmpl = tone_cfg["opener_templates"][hash(title) % len(tone_cfg["opener_templates"])]
    opener = opener_tmpl.format(point=thesis[:80])

    # Build body
    body_lines = []
    for p in points:
        prefix = "→" if tone_cfg["formality"] == "high" else "•"
        body_lines.append(f"{prefix} {p[:120]}")

    # CTA
    cta = tone_cfg["cta"]

    post = f"{opener}\n\n" + "\n".join(body_lines) + f"\n\n{cta}"

    if len(post) > PLATFORM_LIMITS["linkedin"]:
        post = post[:PLATFORM_LIMITS["linkedin"] - 3] + "..."
    return post


def generate_twitter_thread(title: str, body: str, tone: str = DEFAULT_TONE) -> List[str]:
    """Generate a Twitter/X thread (4-6 tweets, each ≤ 280 chars)."""
    tone_cfg = TONES[tone]
    points = extract_key_points(body, max_points=4)
    hooks = extract_hooks(title, body)

    tweets = []

    # Tweet 1: hook
    hook = hooks[0] if hooks else title
    tweet1 = f"1/ {hook}"
    if tone_cfg["emoji_density"] > 0:
        tweet1 = f"1/ 🧵 {hook}"
    if len(tweet1) > 280:
        tweet1 = tweet1[:277] + "..."
    tweets.append(tweet1)

    # Middle tweets: key points
    for i, point in enumerate(points):
        num = i + 2
        tweet = f"{num}/ {point[:250]}"
        if len(tweet) > 280:
            tweet = tweet[:277] + "..."
        tweets.append(tweet)

    # Final tweet: CTA
    final_num = len(tweets) + 1
    cta = tone_cfg["cta"]
    final = f"{final_num}/ {cta}"
    tweets.append(final)

    return tweets


def generate_email_snippet(title: str, body: str, tone: str = DEFAULT_TONE) -> Dict[str, str]:
    """Generate an email subject line + teaser body."""
    tone_cfg = TONES[tone]
    thesis = extract_thesis(body)

    # Subject line
    if tone_cfg["formality"] == "high":
        subject = f"Insight: {title[:50]}"
    elif tone == "urgent":
        subject = f"Don't miss: {title[:45]}"
    elif tone == "witty":
        subject = f"Plot twist about {title.split()[0].lower()}..."
    else:
        subject = f"New: {title[:52]}"

    if len(subject) > PLATFORM_LIMITS["email_subject"]:
        subject = subject[:PLATFORM_LIMITS["email_subject"] - 3] + "..."

    # Teaser body
    teaser = f"{thesis[:150]}\n\n→ Read the full article"
    if len(teaser) > PLATFORM_LIMITS["email_body"]:
        teaser = teaser[:PLATFORM_LIMITS["email_body"] - 3] + "..."

    return {"subject": subject, "body": teaser}


def generate_instagram(title: str, body: str, tone: str = DEFAULT_TONE) -> str:
    """Generate an Instagram caption with emoji and hashtags."""
    tone_cfg = TONES[tone]
    thesis = extract_thesis(body)
    points = extract_key_points(body, max_points=2)

    # Emoji opener
    emojis = ["✨", "💡", "🔥", "🎯", "🚀"]
    emoji = emojis[hash(title) % len(emojis)]

    if tone == "witty":
        caption = f"{emoji} Plot twist: {thesis[:100]}\n\n"
    elif tone == "urgent":
        caption = f"{emoji} Stop scrolling. {thesis[:100]}\n\n"
    elif tone == "casual":
        caption = f"{emoji} Real talk — {thesis[:100].lower()}\n\n"
    else:
        caption = f"{emoji} {thesis[:120]}\n\n"

    for p in points:
        caption += f"→ {p[:80]}\n"

    # Hashtags
    tags = ["#content", "#socialmedia", "#marketing", "#2025"]
    caption += "\n" + " ".join(tags[:tone_cfg["hashtag_count"] + 2])

    if len(caption) > PLATFORM_LIMITS["instagram"]:
        caption = caption[:PLATFORM_LIMITS["instagram"] - 3] + "..."
    return caption


print("Rule-based generators defined: summary, linkedin, twitter_thread, email, instagram")


Rule-based generators defined: summary, linkedin, twitter_thread, email, instagram


## Baseline Test — Article 1, Professional Tone

Let's see all five outputs for the first article using the default `professional` tone.

In [9]:
a = ARTICLES[0]
tone = "professional"

print(f"SOURCE: {a['title']}")
print(f"TONE:   {tone}")
print()

# Summary
print("━" * 70)
print("📝 SHORT SUMMARY")
print("━" * 70)
summary = generate_summary(a["title"], a["body"], tone)
print(summary)
print(f"  [{len(summary)} chars]")

# LinkedIn
print()
print("━" * 70)
print("💼 LINKEDIN POST")
print("━" * 70)
li = generate_linkedin(a["title"], a["body"], tone)
print(li)
print(f"  [{len(li)} chars]")

# Twitter thread
print()
print("━" * 70)
print("🐦 TWITTER/X THREAD")
print("━" * 70)
thread = generate_twitter_thread(a["title"], a["body"], tone)
for t in thread:
    print(t)
    print(f"  [{len(t)} chars]")
    print()

# Email
print("━" * 70)
print("📧 EMAIL SNIPPET")
print("━" * 70)
email = generate_email_snippet(a["title"], a["body"], tone)
print(f"Subject: {email['subject']}")
print(f"  [{len(email['subject'])} chars]")
print(f"Body: {email['body']}")

# Instagram
print()
print("━" * 70)
print("📸 INSTAGRAM CAPTION")
print("━" * 70)
ig = generate_instagram(a["title"], a["body"], tone)
print(ig)
print(f"  [{len(ig)} chars]")


SOURCE: Why Every Developer Should Learn Prompt Engineering in 2025
TONE:   professional

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📝 SHORT SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Why Every Developer Should Learn Prompt Engineering in 2025. But raw model power means nothing without the ability to extract value from it. Key insight: Prompting is an experimental discipline. You test hypotheses, not guess. Add a s.
  [235 chars]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💼 LINKEDIN POST
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
But raw model power means nothing without the ability to extract value from it. — and the implications are significant.

→ Prompting is an experimental discipline. You test hypotheses, not guess. Add a step-by-step instruction and compare outp
→ Context is leverage. A 2000-token context with background information will outperform a 10k-token

## Tone Comparison — Same Article, Four Tones

This is the key demonstration: we hold the article constant and vary only the `tone` parameter. Watch how vocabulary, hook style, and sentence structure change.

In [10]:
a = ARTICLES[1]  # Work-life balance article

print(f"SOURCE: {a['title']}\n")

for tone_name in TONES:
    print(f"{'=' * 70}")
    print(f"TONE: {tone_name.upper()}")
    print(f"{'=' * 70}")

    print(f"\n  Summary: {generate_summary(a['title'], a['body'], tone_name)[:120]}...")
    print(f"\n  LinkedIn opener: {generate_linkedin(a['title'], a['body'], tone_name).split(chr(10))[0][:120]}...")

    thread = generate_twitter_thread(a["title"], a["body"], tone_name)
    print(f"\n  Thread tweet 1: {thread[0][:120]}...")

    email = generate_email_snippet(a["title"], a["body"], tone_name)
    print(f"\n  Email subject: {email['subject']}")

    print(f"\n  Instagram opener: {generate_instagram(a['title'], a['body'], tone_name).split(chr(10))[0][:120]}...")
    print()


SOURCE: The Burnout Myth: Why Work-Life Balance Is Not About Balance

TONE: PROFESSIONAL

  Summary: The Burnout Myth: Why Work-Life Balance Is Not About Balance. The burnout crisis is not about unequal time allocation. K...

  LinkedIn opener: Here's what the evidence suggests: The burnout crisis is not about unequal time allocation....

  Thread tweet 1: 1/ The Burnout Myth: Why Work-Life Balance Is Not About Balance...

  Email subject: Insight: The Burnout Myth: Why Work-Life Balance Is Not Abo

  Instagram opener: 💡 The burnout crisis is not about unequal time allocation....

TONE: CASUAL

  Summary: The Burnout Myth: Why Work-Life Balance Is Not About Balance — the burnout crisis is not about unequal time allocation. ...

  LinkedIn opener: Real talk: The burnout crisis is not about unequal time allocation....

  Thread tweet 1: 1/ 🧵 The Burnout Myth: Why Work-Life Balance Is Not About Balance...

  Email subject: New: The Burnout Myth: Why Work-Life Balance Is Not About

  Insta

## LLM-Enhanced Generation (Ollama)

When Ollama is available, we use structured prompts for more creative, human-sounding output. The prompts encode the same platform constraints as the rule-based generators — but let the LLM handle vocabulary, phrasing, and flow.

**Prompt design principles shown here:**
1. **Role assignment:** "You are a LinkedIn copywriter"
2. **Explicit constraints:** character limits, format requirements
3. **Tone injection:** the tone parameter is inserted directly into the prompt
4. **Output format specification:** tell the model exactly what structure to return

In [11]:
def call_ollama(prompt: str, temperature: float = 0.7) -> str:
    """Call Ollama API with a prompt. Returns empty string on failure."""
    payload = {
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": temperature, "num_predict": 512},
    }
    req = urllib.request.Request(
        f"{OLLAMA_HOST}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        raw = data.get("response", "").strip()
        # Strip <think>...</think> blocks from reasoning models
        raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
        return raw
    except Exception as e:
        print(f"  [Ollama error: {e}]")
        return ""


# Prompt templates — note how each encodes platform-specific constraints
LLM_PROMPTS = {
    "summary": (
        "Write a 2-3 sentence summary of this article. "
        "Tone: {tone}. No emoji. No hashtags. Just the core idea.\n\n"
        "Title: {title}\nArticle: {excerpt}\n\nSummary:"
    ),
    "linkedin": (
        "You are a LinkedIn copywriter. Write a LinkedIn post about this article.\n\n"
        "RULES:\n"
        "- Tone: {tone}\n"
        "- 3-4 short paragraphs with line breaks\n"
        "- End with a question to drive comments\n"
        "- No hashtags\n"
        "- Under 1300 characters\n\n"
        "Title: {title}\nArticle: {excerpt}\n\nLinkedIn post:"
    ),
    "twitter_thread": (
        "Write a 4-5 tweet Twitter/X thread about this article.\n\n"
        "RULES:\n"
        "- Tone: {tone}\n"
        "- Each tweet MUST be under 280 characters\n"
        "- Number tweets as 1/ 2/ 3/ etc\n"
        "- First tweet hooks the reader\n"
        "- Last tweet asks for engagement\n\n"
        "Title: {title}\nArticle: {excerpt}\n\nThread:"
    ),
    "email": (
        "Write an email subject line and 2-sentence teaser for a newsletter.\n\n"
        "RULES:\n"
        "- Tone: {tone}\n"
        "- Subject line under 60 characters\n"
        "- Teaser: exactly 2 sentences creating curiosity\n"
        "- Format: Subject: [line]\nTeaser: [text]\n\n"
        "Title: {title}\nArticle: {excerpt}\n\nEmail:"
    ),
    "instagram": (
        "Write an Instagram caption for a post about this article.\n\n"
        "RULES:\n"
        "- Tone: {tone}\n"
        "- Start with an emoji\n"
        "- 2-3 short paragraphs\n"
        "- End with 3-5 relevant hashtags\n"
        "- Under 2200 characters\n\n"
        "Title: {title}\nArticle: {excerpt}\n\nCaption:"
    ),
}

print("LLM prompt templates defined for:", list(LLM_PROMPTS.keys()))
if not USE_LLM:
    print("  (Ollama not available — these won't be used in this run)")


LLM prompt templates defined for: ['summary', 'linkedin', 'twitter_thread', 'email', 'instagram']
  (Ollama not available — these won't be used in this run)


## Full Pipeline — Repurpose an Article

This is the main function that orchestrates conversion. It uses LLM generation when available and falls back to rule-based extraction otherwise.

In [12]:
def repurpose_article(article: Dict[str, str],
                       tone: str = DEFAULT_TONE) -> Dict[str, Any]:
    """Convert a blog article into all 5 platform formats."""
    title = article["title"]
    body = article["body"]
    excerpt = body[:500]  # First 500 chars as LLM context

    result = {"title": title, "tone": tone, "method": "rule-based"}

    if USE_LLM:
        result["method"] = "LLM"
        for platform, tmpl in LLM_PROMPTS.items():
            prompt = tmpl.format(title=title, excerpt=excerpt, tone=tone)
            llm_out = call_ollama(prompt, temperature=0.7)
            if llm_out:
                result[platform] = llm_out
            else:
                # Fallback to rule-based if LLM fails
                result[platform] = _rule_based_fallback(platform, title, body, tone)
                result["method"] = "LLM (partial fallback)"
    else:
        result["summary"] = generate_summary(title, body, tone)
        result["linkedin"] = generate_linkedin(title, body, tone)
        thread = generate_twitter_thread(title, body, tone)
        result["twitter_thread"] = "\n\n".join(thread)
        email = generate_email_snippet(title, body, tone)
        result["email"] = f"Subject: {email['subject']}\n{email['body']}"
        result["instagram"] = generate_instagram(title, body, tone)

    return result


def _rule_based_fallback(platform: str, title: str, body: str, tone: str) -> str:
    """Fallback when LLM fails for a specific platform."""
    if platform == "summary":
        return generate_summary(title, body, tone)
    elif platform == "linkedin":
        return generate_linkedin(title, body, tone)
    elif platform == "twitter_thread":
        return "\n\n".join(generate_twitter_thread(title, body, tone))
    elif platform == "email":
        e = generate_email_snippet(title, body, tone)
        return f"Subject: {e['subject']}\n{e['body']}"
    elif platform == "instagram":
        return generate_instagram(title, body, tone)
    return ""


# Run pipeline on all articles with different tones
results = []
test_configs = [
    (0, "professional"),
    (1, "casual"),
    (2, "urgent"),
]

for idx, tone in test_configs:
    a = ARTICLES[idx]
    print(f"\nProcessing: {a['title'][:50]}... (tone={tone})")
    r = repurpose_article(a, tone=tone)
    results.append(r)
    print(f"  Method: {r['method']}")
    print(f"  Platforms: {[k for k in r if k not in ('title','tone','method')]}")



Processing: Why Every Developer Should Learn Prompt Engineerin... (tone=professional)
  Method: rule-based
  Platforms: ['summary', 'linkedin', 'twitter_thread', 'email', 'instagram']

Processing: The Burnout Myth: Why Work-Life Balance Is Not Abo... (tone=casual)
  Method: rule-based
  Platforms: ['summary', 'linkedin', 'twitter_thread', 'email', 'instagram']

Processing: The AI Safety Paradox: Why Slowing Down Makes Depl... (tone=urgent)
  Method: rule-based
  Platforms: ['summary', 'linkedin', 'twitter_thread', 'email', 'instagram']


## Output Display — All Generated Content

In [13]:
for r in results:
    print(f"\n{'═' * 70}")
    print(f"ARTICLE: {r['title'][:60]}")
    print(f"TONE: {r['tone']}  |  METHOD: {r['method']}")
    print(f"{'═' * 70}")

    platforms = {
        "summary": ("📝", "SHORT SUMMARY"),
        "linkedin": ("💼", "LINKEDIN POST"),
        "twitter_thread": ("🐦", "TWITTER/X THREAD"),
        "email": ("📧", "EMAIL SNIPPET"),
        "instagram": ("📸", "INSTAGRAM CAPTION"),
    }

    for key, (icon, label) in platforms.items():
        if key in r:
            print(f"\n{icon} {label}")
            print("─" * 50)
            print(r[key])
            print(f"  [{len(r[key])} chars]")



══════════════════════════════════════════════════════════════════════
ARTICLE: Why Every Developer Should Learn Prompt Engineering in 2025
TONE: professional  |  METHOD: rule-based
══════════════════════════════════════════════════════════════════════

📝 SHORT SUMMARY
──────────────────────────────────────────────────
Why Every Developer Should Learn Prompt Engineering in 2025. But raw model power means nothing without the ability to extract value from it. Key insight: Prompting is an experimental discipline. You test hypotheses, not guess. Add a s.
  [235 chars]

💼 LINKEDIN POST
──────────────────────────────────────────────────
But raw model power means nothing without the ability to extract value from it. — and the implications are significant.

→ Prompting is an experimental discipline. You test hypotheses, not guess. Add a step-by-step instruction and compare outp
→ Context is leverage. A 2000-token context with background information will outperform a 10k-token context with noi

## Evaluation — Constraint Checking

We verify that each generated output meets its platform's hard constraints. This is the quality gate before any content would be published.

In [14]:
def evaluate_output(result: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    """Check platform constraints for generated content."""
    checks = {}

    # Summary
    if "summary" in result:
        text = result["summary"]
        checks["summary"] = {
            "length_ok": len(text) <= PLATFORM_LIMITS["summary"],
            "length": len(text),
            "has_content": len(text) > 20,
        }

    # LinkedIn
    if "linkedin" in result:
        text = result["linkedin"]
        checks["linkedin"] = {
            "length_ok": len(text) <= PLATFORM_LIMITS["linkedin"],
            "length": len(text),
            "has_paragraphs": text.count("\n") >= 2,
            "has_cta": "?" in text[-100:] if len(text) > 100 else "?" in text,
        }

    # Twitter thread
    if "twitter_thread" in result:
        text = result["twitter_thread"]
        tweets = [t for t in text.split("\n\n") if t.strip()]
        over_limit = [t for t in tweets if len(t) > PLATFORM_LIMITS["twitter_tweet"]]
        checks["twitter_thread"] = {
            "tweet_count": len(tweets),
            "all_under_280": len(over_limit) == 0,
            "over_limit_count": len(over_limit),
            "has_numbering": any(t.strip().startswith("1/") for t in tweets),
        }

    # Email
    if "email" in result:
        text = result["email"]
        has_subject = "Subject:" in text or "subject:" in text.lower()
        subject_line = ""
        if has_subject:
            for line in text.split("\n"):
                if line.lower().startswith("subject:"):
                    subject_line = line.split(":", 1)[1].strip()
                    break
        checks["email"] = {
            "has_subject_line": has_subject,
            "subject_length": len(subject_line),
            "subject_under_60": len(subject_line) <= PLATFORM_LIMITS["email_subject"],
        }

    # Instagram
    if "instagram" in result:
        text = result["instagram"]
        checks["instagram"] = {
            "length_ok": len(text) <= PLATFORM_LIMITS["instagram"],
            "length": len(text),
            "has_hashtags": "#" in text,
            "has_emoji": any(ord(c) > 127 for c in text),
        }

    return checks


print("EVALUATION REPORT")
print("=" * 70)
for r in results:
    checks = evaluate_output(r)
    print(f"\nArticle: {r['title'][:50]}... (tone={r['tone']})")

    total_pass = 0
    total_checks = 0
    for platform, platform_checks in checks.items():
        passed = sum(1 for v in platform_checks.values() if v is True)
        total = sum(1 for v in platform_checks.values() if isinstance(v, bool))
        total_pass += passed
        total_checks += total

        status = "✓" if passed == total else "⚠"
        print(f"  {status} {platform}: {passed}/{total} checks passed")
        for check_name, check_val in platform_checks.items():
            if isinstance(check_val, bool):
                icon = "✓" if check_val else "✗"
                print(f"      {icon} {check_name}")
            else:
                print(f"      · {check_name} = {check_val}")

    print(f"  TOTAL: {total_pass}/{total_checks}")


EVALUATION REPORT

Article: Why Every Developer Should Learn Prompt Engineerin... (tone=professional)
  ✓ summary: 2/2 checks passed
      ✓ length_ok
      · length = 235
      ✓ has_content
  ✓ linkedin: 3/3 checks passed
      ✓ length_ok
      · length = 542
      ✓ has_paragraphs
      ✓ has_cta
  ✓ twitter_thread: 2/2 checks passed
      · tweet_count = 6
      ✓ all_under_280
      · over_limit_count = 0
      ✓ has_numbering
  ✓ email: 2/2 checks passed
      ✓ has_subject_line
      · subject_length = 59
      ✓ subject_under_60
  ✓ instagram: 3/3 checks passed
      ✓ length_ok
      · length = 271
      ✓ has_hashtags
      ✓ has_emoji
  TOTAL: 12/12

Article: The Burnout Myth: Why Work-Life Balance Is Not Abo... (tone=casual)
  ✓ summary: 2/2 checks passed
      ✓ length_ok
      · length = 189
      ✓ has_content
  ✓ linkedin: 3/3 checks passed
      ✓ length_ok
      · length = 298
      ✓ has_paragraphs
      ✓ has_cta
  ✓ twitter_thread: 2/2 checks passed
      · tweet_

## Error Analysis

### Rule-based generation weaknesses

| Issue | Example | Root Cause |
|-------|---------|-----------|
| **Repetitive openers** | Every LinkedIn post starts with the same template | Limited opener variety (3 templates per tone) |
| **No creative rewriting** | Points are truncated, not rephrased | Extraction-only approach — no synthesis |
| **Tone is shallow** | "casual" just lowercases and adds emoji | No vocabulary substitution or sentence restructuring |
| **Thread tweets have uneven length** | Some tweets use 50 chars, others 280 | No length balancing across thread |

### LLM generation weaknesses (when available)

| Issue | Example | Root Cause |
|-------|---------|-----------|
| **Character limit violations** | Twitter tweet exceeds 280 chars | Models struggle with exact counting |
| **Inconsistent formatting** | Thread numbering varies (1/, 1., Tweet 1:) | Output format not sufficiently constrained |
| **Tone drift** | Starts casual, becomes formal mid-post | Long outputs lose prompt influence |
| **Hallucinated claims** | Adds statistics not in source article | Model generates plausible but fabricated details |

### Mitigation strategies

1. **Post-generation validation** — check constraints and regenerate failures
2. **Few-shot examples** — show the model exactly what good output looks like
3. **Output parsing** — extract structured fields instead of trusting free-form output
4. **Human-in-the-loop** — flag low-confidence outputs for manual review

## Limitations

1. **No visual content generation** — social posts often need images, carousels, or video thumbnails; we generate text only
2. **No audience-specific adaptation** — we vary tone but not target demographic (developer vs. executive vs. student)
3. **No performance feedback** — no mechanism to learn which adaptations get more engagement
4. **Fixed tone definitions** — real brand voices are more nuanced than four presets
5. **No A/B variant generation** — production systems generate 3-5 variants per platform and test
6. **English only** — no multilingual support; cross-language adaptation is a separate challenge
7. **No scheduling or posting** — this is generation only, not a publishing pipeline

## How to Improve This Project

### Short-term improvements
1. **Add few-shot examples to LLM prompts** — include 2-3 example outputs per platform to stabilize format
2. **Post-generation retry loop** — if output exceeds char limit, regenerate with stricter prompt
3. **Vocabulary substitution** — build tone-specific word maps ("important" → "crucial" / "huge" / "game-changing")

### Medium-term improvements
4. **Brand voice profiles** — let users define custom tone configurations beyond the four presets
5. **Image suggestion** — extract keywords and suggest stock photo queries for each post
6. **Multi-article digests** — combine 3 articles into one weekly roundup per platform
7. **Engagement prediction** — train a classifier on historical post performance to rank variants

### Production-grade extensions
8. **Buffer / Hootsuite API integration** — schedule posts directly from the pipeline
9. **Analytics feedback loop** — pull engagement metrics back to improve generation
10. **Team collaboration** — approval workflows, edit tracking, version history per post

## Production Considerations

| Concern | Approach |
|---------|----------|
| **Latency** | Rule-based: <100ms per article. LLM: 2-5s per platform. Pre-generate during low-traffic hours. |
| **Cost** | Self-hosted Ollama: $0. API-based (GPT-4): ~$0.05 per article × 5 platforms = $0.25/article. |
| **Quality assurance** | Automated constraint checks + human review for high-stakes content. |
| **Brand safety** | Add keyword blocklists and sentiment guards before publishing. |
| **Rate limiting** | Platform APIs enforce posting limits. Queue and schedule, don't burst. |
| **Data privacy** | Internal blog content may be confidential. Self-hosted LLM preferred over cloud APIs. |
| **Monitoring** | Track character limit violations, tone drift, and engagement metrics per platform. |

## Common Mistakes

### 1. Cross-posting without adaptation
**Wrong:** Copy the LinkedIn post verbatim to Twitter.
**Why it fails:** LinkedIn rewards long-form credibility. Twitter rewards snappy takes. A 1 000-char LinkedIn post truncated to 280 chars makes no sense.
**Fix:** Use platform-specific generators — never cross-post raw.

### 2. Ignoring character limits
**Wrong:** "The model will figure out the right length."
**Why it fails:** LLMs cannot count characters. They approximate, and approximation on a 280-char limit means 40% of tweets will overflow.
**Fix:** Always post-validate with `len(text) <= limit`. Regenerate failures.

### 3. Same tone everywhere
**Wrong:** Using "professional" tone on TikTok and Instagram.
**Why it fails:** Platform audiences have evolved distinct communication norms. Formal language on casual platforms reads as out-of-touch.
**Fix:** Match tone to platform culture. Professional ≠ better; appropriate = better.

### 4. Forgetting the hook
**Wrong:** Starting a Twitter thread with "In this article, we discuss…"
**Why it fails:** Social feeds are competitive. Users scroll past anything that doesn't grab attention in 3 seconds.
**Fix:** Lead with a surprising claim, contrarian take, or emotional trigger.

### 5. Over-hashtagging
**Wrong:** "#AI #MachineLearning #DeepLearning #NLP #GPT #LLM #Tech #Coding #Developer"
**Why it fails:** Platforms penalize hashtag spam. LinkedIn shows 3-5 hashtags optimally. Instagram tolerates more but relevance matters.
**Fix:** Use 2-5 highly relevant hashtags per platform.

### 6. Not testing with real articles
**Wrong:** Building the pipeline on one sample article and shipping.
**Why it fails:** Different article structures (listicles vs. narrative vs. opinion) break different extractors.
**Fix:** Test on at least 10 diverse articles before deploying.

## Mini Challenge / Exercises

### Challenge 1: Add a new tone — "storytelling"
Create a fifth tone configuration that uses:
- First-person narrative ("I remember when…")
- Anecdotes and scene-setting
- Emotional arc (problem → struggle → insight)

Add it to `TONES` and regenerate all outputs. How does it change the LinkedIn post?

### Challenge 2: Reddit adapter
Build a `generate_reddit()` function for r/ExplainLikeImFive style:
- Simple language, no jargon
- Analogy-heavy explanations
- Question-and-answer format

What prompt changes are needed versus LinkedIn?

### Challenge 3: Engagement scoring
Write a `score_engagement_potential(text, platform)` function that estimates:
- Hook strength (0-10): does the opener grab attention?
- Length fit (0-10): is it optimal length for the platform?
- CTA clarity (0-10): does it tell the reader what to do next?

Test it on all generated outputs. Which platform/tone combination scores highest?

### Challenge 4: Multi-article digest
Modify the pipeline to take 3 articles and produce:
- A single LinkedIn "weekly roundup" post
- A single email newsletter with all 3 teasers
- A single Twitter thread covering all 3 topics

How does your extraction strategy change when combining multiple sources?

## Final Summary / Key Takeaways

### What we built
A complete **blog-to-social content repurposing pipeline** that converts one long-form article into five platform-specific formats, each with configurable tone/style controls.

### Key engineering insights

1. **Prompt adaptation is not just length compression.** Each platform requires changes across five dimensions: length, hook style, tone, structure, and CTA. Changing one dimension without the others produces cringe content.

2. **Tone is orthogonal to platform.** You can be professional on Twitter or casual on LinkedIn. The tone parameter modifies vocabulary and sentence structure independently of platform constraints.

3. **Rule-based generation sets the floor.** Deterministic extraction is fast, predictable, and debuggable. It produces serviceable content that meets all constraints. LLM generation raises the ceiling — more creative, more natural — but introduces unpredictability.

4. **Post-generation validation is mandatory.** LLMs cannot count characters. Never trust that generated content meets constraints without checking. Build a validation layer that rejects and regenerates.

5. **Platform norms evolve.** What works on LinkedIn in 2025 may not work in 2026. The pipeline should be designed for easy prompt updates, not hardcoded rules.

### Architecture pattern
```
Source content
  → Extract (thesis, key points, hooks)  ← deterministic
    → Adapt (platform × tone matrix)     ← rule-based or LLM
      → Validate (constraints)           ← deterministic
        → Publish or regenerate
```

This Extract → Adapt → Validate pattern applies broadly to any content transformation task.